<a href="https://colab.research.google.com/github/prkalva10/IA_Machine_Learning/blob/main/IA_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
#importações do dataset
import os
import kagglehub
import pandas as pd
import numpy as np

#importaçoes de gráfico
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Download latest version
path = kagglehub.dataset_download("yashdevladdha/uber-ride-analytics-dashboard")

df = pd.read_csv(os.path.join(path, 'ncr_ride_bookings.csv'))

#print(df.head())

plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

df['Booking ID'] = df['Booking ID'].str.replace('"','', regex=False)
df['Customer ID'] = df['Customer ID'].str.replace('"','', regex=False)

df['Date'] = pd.to_datetime(df['Date'])
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S')

print(f"DataSet carregado: {df.shape[0]:,} corridas.")

Using Colab cache for faster access to the 'uber-ride-analytics-dashboard' dataset.
DataSet carregado: 150,000 corridas.


In [13]:
# Mantendo apenas corridas completadas

df_modelo = df[df['Booking Status'] == "Completed"].copy()

df_modelo = df_modelo.reset_index(drop=True)

print("Total de corridas")
print(len(df))
print("Concluídas")
print(len(df_modelo))
print("Descartadas")
print(len(df) - len(df_modelo))


Total de corridas
150000
Concluídas
93000
Descartadas
57000


In [15]:
# Feature Engenier

df_modelo['hora'] = df_modelo['Time'].dt.hour
df_modelo['dia_semana'] = df_modelo['Date'].dt.dayofweek
df_modelo['mes'] = df_modelo['Date'].dt.month

#print(df_modelo.head())

# Final de semana: 1=SIM, 0 = Não
df_modelo['final_semana'] = df_modelo['dia_semana'].isin([5,6]).astype(int)

# Hora do rush: (7-10h ou 17-20h)
df_modelo['hora_rush'] = (
    df_modelo['hora'].between(7,10) |
   df_modelo['hora'].between(17,20)
).astype(int)

print("Corridas no fim de semana: ",
      df_modelo['final_semana'].sum(),
      f'({100*df_modelo['final_semana'].mean():.1f}%)')

print("Corridas em hora de pico: ",
      df_modelo['hora_rush'].sum(),
      f'({100*df_modelo["hora_rush"].mean():.1f}%)')

Corridas no fim de semana:  26756 (28.8%)
Corridas em hora de pico:  45960 (49.4%)


In [18]:
# Consistencia

df_modelo = df_modelo[(df_modelo['Ride Distance'] > 0) &
                      (df_modelo['Booking Value'] > 0)].copy()

print(f"Após remover valores impossíveis: {len(df_modelo):,} corridas")

Após remover valores impossíveis: 93,000 corridas


In [20]:
def limites_iqr(series):

  q1 = series.quantile(0.25)
  q3 = series.quantile(0.75)
  iqr = q3 - q1
  res = q1 - 1.5 * iqr, q3 + 1.5 * iqr
  return res

val_inf, val_sup = limites_iqr(df_modelo['Booking Value'])
dist_inf, dist_sup = limites_iqr(df_modelo['Ride Distance'])

print("Limites calculados")
print(f"Booking value: R$ {val_inf:.0f} - R$ {val_sup:.0f}")
print(f"Ride Distance: {dist_inf:.1f} - {dist_sup:.1f}")

mask = (
    df_modelo["Booking Value"].between(val_inf, val_sup) &
    df_modelo["Ride Distance"].between(dist_inf, dist_sup)
)

df_limpo = df_modelo[mask].copy().reset_index(drop=True)

print(f"Corridas antes limpeza: {len(df_modelo):,}")
print(f"Corridas após limpeza: {len(df_limpo):,}")

Limites calculados
Booking value: R$ -448 - R$ 1372
Ride Distance: -21.7 - 73.7
Corridas antes limpeza: 93,000
Corridas após limpeza: 89,873
